# 08 - MCDM Baselines (`AHP-TOPSIS`, `AHP-ELECTRE I`, `AHP-WP`, `Equal-SAW`), 7-Variant Identity Ablation & Controlled Seating Perturbation (WORKING COPY - TARGETED FIX)
**Master Workspace:** `MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/08_COLAB_NOTEBOOKS`  
**Original Notebook Preserved:** `08_Baseline_Ablation_Analysis.ipynb`  

### Targeted Fixes Applied in This Notebook:
1. **Added MCDM Concordance & Holm-Bonferroni Summary Export:** Integrated `compute_mcdm_concordance_holm_summary()` in `Cell 3` to compute and export Spearman $\rho$, Fisher-$z$ 95% CIs, Kendall $\tau$, Mean/Max rank shifts, Top-1/Top-5 overlap, and step-down Holm-Bonferroni adjusted $p$-values (`mcdm_topsis_electre_concordance_summary.csv`).
2. **Fixed `NameError: ABLATION_VARIANTS` (`BUG-01`):** Exported both `ABLATION_CONFIG_OVERRIDES` and `ABLATION_VARIANTS` in `Cell 5`.
3. **Fixed All 7 Identity Ablation Variants (`BUG-08`):**
   - Variant 2 (`2_Without_Temporal_Evidence`) now hooks `_track_quality_from_scores` to set `purity_agg_weight = 0.0` (removing the `0.45 * purity` dominance term) in addition to `weight_temporal = 0.0`.
   - Variant 3 (`3_Without_Appearance_Evidence`) explicitly checks whether external ReID similarity features exist; when `reid_similarity_df` is absent (as in cached `detections.csv` with `with_reid=False`), it flags `REQUIRES_EXTERNAL_REID_FEATURES_OR_VIDEO_REINFERENCE` rather than fabricating a no-op duplicate of Variant 1.
   - Variant 7 (`7_Without_Long_Gap_Reassociation`) actively identifies the 12 long-gap track fragments (`intertrack_gap_frames > 90` frames) via `merge_track_fragments()` and revokes their seat assignments (`unassigned`) before recomputing frame conflict resolution and validation metrics.
4. **Added Controlled Seating Arrangement Perturbation Suite (Section 3 Reviewer Cross-Scenario Response):** Added `generate_perturbed_seat_map()` and `run_controlled_seating_perturbation_suite()` in `Cell 5` & `Cell 7` to test spatial seat jitter (`5 px` to `50 px`), ROI scaling (`50%`, `75%`, `125%`), row/column pitch compression (`15%`), and adjacent seat swaps while keeping actual observations (`ACTUAL_OBSERVATION`) strictly separated from simulations (`CONTROLLED_SIMULATION`).


In [1]:
# ==============================================================================
# CENTRALIZED GOOGLE DRIVE PATHS, EXECUTION LOCKS & MULTI-DETECTOR SELECTOR
# ==============================================================================
import json
import logging
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Select Active Detector Model for MCDM, Identity Ablation & Controlled Seating Perturbation:
#   "yolo11m"       -> Primary YOLO11m baseline
#   "yolov8m"       -> YOLOv8m baseline (reads/builds from multi_detector_pipeline/yolov8m/)
#   "rfdetr_small"  -> RF-DETR-Small ("S", reads/builds from multi_detector_pipeline/rfdetr_small/)
#   "rfdetr_large"  -> RF-DETR-Large ("L", reads/builds from multi_detector_pipeline/rfdetr_large/)
#   "ALL_AVAILABLE" -> Automatically builds missing matrices from any existing detections.csv and runs all ready detectors!
ACTIVE_DETECTOR_MODEL = "ALL_AVAILABLE"

# Set AUTO_RUN_MISSING_VIDEO_INFERENCE = True if you want Notebook 08 to also run 23,471-frame
# video inference automatically when detections.csv has not been generated in Notebook 04 yet
AUTO_RUN_MISSING_VIDEO_INFERENCE = False

ENABLE_MCDM_EVALUATION = True
ENABLE_IDENTITY_ABLATION = True
ENABLE_SEAT_PERTURBATION_SIMULATION = True

ENABLE_ROBOFLOW_STAGING_DOWNLOAD = False
ENABLE_DATASET_REPLACEMENT = False
ENABLE_TRAINING = False
ENABLE_MODEL_EVALUATION = False
ENABLE_FULL_VIDEO_INFERENCE = False
ENABLE_TRACKER_REPLAY = False
ENABLE_TEMPORAL_EXECUTION = False
ENABLE_MONTE_CARLO = False
ENABLE_GPU_EXPERIMENTS = False

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s")
logger = logging.getLogger("YOLO11_IJITCS_SyncedColab")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

MASTER_ROOT = Path(
    "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026"
)
DATASET_ROOT = MASTER_ROOT / "03_DATASET_AUDIT" / "YOLO11_DATASET_UNZIPPED"
DATA_YAML_PATH = DATASET_ROOT / "data_colab.yaml"
REPO_ROOT = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "YOLO11_Repository"
MODULES_DIR = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "Reconstructed_Research_Modules"
VIDEO_PATH = (
    MASTER_ROOT
    / "02_ORIGINAL_RESEARCH_ARCHIVE"
    / "Original_Videos"
    / "video_part001.mp4"
)
CHECKPOINT_DIR = MASTER_ROOT / "04_MODEL_AND_CONFIGURATION" / "Checkpoints"
OUTPUT_DIR = MASTER_ROOT / "09_REVALIDATED_EXPERIMENT_OUTPUTS"
OUTPUT_PATH = OUTPUT_DIR
MULTI_DETECTOR_PIPELINE_DIR = OUTPUT_DIR / "multi_detector_pipeline"

for _p in (REPO_ROOT, MODULES_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
importlib.invalidate_caches()
for _m in list(sys.modules.keys()):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]

from src.paths_config import get_detector_pipeline_paths
from src.detector_adapters_v2_aligned import ensure_detector_downstream_csvs_ready

def _resolve_existing(candidates: list[Path]) -> Path:
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]

TRACKING_VAL_DIR = MASTER_ROOT / "06_TRACKING_AND_IDENTITY_VALIDATION"
AHP_SAW_VAL_DIR = MASTER_ROOT / "07_AHP_SAW_AND_STATISTICAL_VALIDATION"
STUDENT_SEATS_PATH = _resolve_existing([
    REPO_ROOT / "data" / "student_seats.csv",
    TRACKING_VAL_DIR / "student_seats.csv",
])

def get_active_detector_targets(selector: str, auto_build: bool = True) -> list[dict[str, Path]]:
    norm = str(selector).strip().upper()
    candidate_keys = (
        ["yolo11m", "yolov8m", "rfdetr_small", "rfdetr_large"]
        if norm == "ALL_AVAILABLE"
        else [selector]
    )
    targets = []
    for k in candidate_keys:
        if auto_build and MASTER_ROOT.exists():
            prep = ensure_detector_downstream_csvs_ready(
                detector_key=k,
                master_root=MASTER_ROOT,
                seats_csv_path=STUDENT_SEATS_PATH,
                video_path=VIDEO_PATH,
                fps=25.0,
                total_video_frames=23471,
                auto_run_video_inference=AUTO_RUN_MISSING_VIDEO_INFERENCE,
            )
            p = prep["paths"]
            if prep["ready"]:
                targets.append(p)
            else:
                logger.info(
                    "Skipping detector '%s' in Notebook 08 (%s). Run Notebook 04 for '%s' or set AUTO_RUN_MISSING_VIDEO_INFERENCE = True.",
                    p["detector_key"],
                    prep["status"],
                    p["detector_key"],
                )
        else:
            p = get_detector_pipeline_paths(k, master_root=MASTER_ROOT)
            if p["duration_matrix_csv"].exists() or p["detections_csv"].exists():
                targets.append(p)
    return targets or [get_detector_pipeline_paths("yolo11m", master_root=MASTER_ROOT)]

_preview_targets = get_active_detector_targets(ACTIVE_DETECTOR_MODEL, auto_build=False)
DETECTIONS_CSV_PATH = _preview_targets[0]["detections_csv"]
DURATION_MATRIX_CSV_PATH = _preview_targets[0]["duration_matrix_csv"]

logger.info("Active Detector Selector in Notebook 08: %s | Candidates found: %s", ACTIVE_DETECTOR_MODEL, [t["detector_key"] for t in _preview_targets])

Mounted at /content/drive


## Step 2: MCDM Comparative Baselines (`AHP-SAW`, `AHP-TOPSIS`, `AHP-ELECTRE I`, `AHP-WP`, `Equal-SAW`) & Holm-Bonferroni Concordance


In [2]:
# ==============================================================================
# MCDM COMPARATIVE BASELINES (`AHP-SAW`, `AHP-TOPSIS`, `AHP-ELECTRE I`, `AHP-WP`, `Equal-SAW`)
# & HOLM-BONFERRONI CONCORDANCE SUITE (MULTI-DETECTOR SUPPORT)
# ==============================================================================
from src.ahp import ahp_weights
from src.mcdm_topsis_electre_adapters import (
    compute_mcdm_concordance_holm_summary,
    evaluate_all_mcdm_methods,
)
from src.monte_carlo import monte_carlo_sensitivity
from src.saw import saw_score, zero_diagnostics
from src.sensitivity import one_at_a_time
from src.wp import compare_saw_wp, wp_score

if ENABLE_MCDM_EVALUATION:
    _targets = get_active_detector_targets(ACTIVE_DETECTOR_MODEL)
    _all_concordance_rows = []
    for _t in _targets:
        _dkey = _t["detector_key"]
        _dur_path = _t["duration_matrix_csv"]
        _out_dir = OUTPUT_DIR if _dkey == "yolo11m" else _t["pipeline_dir"]
        _out_dir.mkdir(parents=True, exist_ok=True)
        if not _dur_path.exists():
            raise FileNotFoundError(
                f"Duration matrix for '{_dkey}' not found at {_dur_path}. Run Notebook 04 with ACTIVE_DETECTOR_MODEL = '{_dkey}' first."
            )
        df_dur = pd.read_csv(_dur_path)
        df_mcdm_res = evaluate_all_mcdm_methods(df_dur)
        df_mcdm_sum = compute_mcdm_concordance_holm_summary(df_mcdm_res)
        df_mcdm_res.insert(0, "detector_model", _dkey)
        df_mcdm_sum.insert(0, "detector_model", _dkey)

        if _dkey == "yolo11m":
            df_mcdm_res.drop(columns=["detector_model"]).to_csv(_out_dir / "mcdm_topsis_electre_comparison.csv", index=False)
            df_mcdm_sum.drop(columns=["detector_model"]).to_csv(_out_dir / "mcdm_topsis_electre_concordance_summary.csv", index=False)
        df_mcdm_res.to_csv(_out_dir / f"mcdm_topsis_electre_comparison_{_dkey}.csv", index=False)
        df_mcdm_sum.to_csv(_out_dir / f"mcdm_topsis_electre_concordance_summary_{_dkey}.csv", index=False)
        _all_concordance_rows.append(df_mcdm_sum)

        print(f"\n=== MCDM Rankings ({_dkey}) ===")
        print(df_mcdm_res.to_string(index=False))
        print(f"\n=== MCDM Holm-Bonferroni Concordance ({_dkey}) ===")
        print(df_mcdm_sum.to_string(index=False))

    if len(_all_concordance_rows) > 1:
        pd.concat(_all_concordance_rows, ignore_index=True).to_csv(
            MULTI_DETECTOR_PIPELINE_DIR / "multi_detector_mcdm_concordance_summary.csv", index=False
        )
else:
    logger.info(
        "[LOCKED] MCDM evaluation (ACTIVE_DETECTOR_MODEL = '%s') is prepared via "
        "src.mcdm_topsis_electre_adapters and locked (ENABLE_MCDM_EVALUATION = False).",
        ACTIVE_DETECTOR_MODEL,
    )


=== MCDM Rankings (yolo11m) ===
detector_model student_id AHP_SAW_Score  AHP_SAW_Rank AHP_TOPSIS_Score  AHP_TOPSIS_Rank AHP_ELECTRE_NetFlow  AHP_ELECTRE_Rank AHP_WP_Score  AHP_WP_Rank Equal_SAW_Score  Equal_SAW_Rank
       yolo11m        S03      0.715082             1         0.670176                1            1.318004                 1     1.000000            1        0.719721               1
       yolo11m        S21      0.669809             2         0.547692                2            1.074690                 3     0.507445            5        0.649321               3
       yolo11m        S15      0.660878             3         0.546248                3            1.270771                 2     0.715137            3        0.658531               2
       yolo11m        S11      0.555991             4         0.495525                4            0.901361                 4     0.796780            2        0.614290               4
       yolo11m        S19      0.413866        

## Step 3: 7-Variant Fixed-Seat Identity Component Ablation (`ENABLE_IDENTITY_ABLATION = False`)


In [3]:
# ==============================================================================
# 7-VARIANT FIXED-SEAT IDENTITY COMPONENT ABLATION (MULTI-DETECTOR SUPPORT)
# ==============================================================================
import src.identity as base_identity_module
from src.identity_ablation_aligned import (
    ABLATION_CONFIG_OVERRIDES,
    ABLATION_VARIANTS,
    CONTROLLED_PERTURBATION_SCENARIOS,
    run_component_ablation,
    run_controlled_seating_perturbation_suite,
)

if ENABLE_IDENTITY_ABLATION:
    seats_df = base_identity_module.load_seat_map(str(STUDENT_SEATS_PATH), expected_students=21)
    for _t in get_active_detector_targets(ACTIVE_DETECTOR_MODEL):
        _dkey = _t["detector_key"]
        _det_path = _t["detections_csv"]
        _out_dir = OUTPUT_DIR if _dkey == "yolo11m" else _t["pipeline_dir"]
        _out_dir.mkdir(parents=True, exist_ok=True)
        if not _det_path.exists():
            raise FileNotFoundError(
                f"Detections CSV for '{_dkey}' not found at {_det_path}. Run Notebook 04 with ACTIVE_DETECTOR_MODEL = '{_dkey}' first."
            )
        det_df = pd.read_csv(_det_path)
        ablation_df = run_component_ablation(
            detections_df=det_df,
            seats_df=seats_df,
            base_identity_module=base_identity_module,
            reid_similarity_df=None,
            fps=25.0,
            total_video_frames=23471,
            enable_ablation=ENABLE_IDENTITY_ABLATION,
        )
        ablation_df.insert(0, "detector_model", _dkey)
        if _dkey == "yolo11m":
            ablation_df.drop(columns=["detector_model"]).to_csv(_out_dir / "identity_component_ablation_results.csv", index=False)
        ablation_df.to_csv(_out_dir / f"identity_component_ablation_results_{_dkey}.csv", index=False)
        print(f"\n=== 7-Variant Identity Component Ablation ({_dkey}) ===")
        print(ablation_df.to_string(index=False))
else:
    logger.info(
        "[LOCKED] 7-variant Fixed-Seat Identity Component Ablation (ACTIVE_DETECTOR_MODEL = '%s') "
        "is locked (ENABLE_IDENTITY_ABLATION = False).",
        ACTIVE_DETECTOR_MODEL,
    )


=== 7-Variant Identity Component Ablation (yolo11m) ===
detector_model                               variant                                     execution_status  reid_features_available  reject_long_gap  n_primary_assigned  primary_assignment_rate_pct  n_assigned_students  mean_seat_purity_preprop_pct  mean_student_frame_coverage_pct  n_unassigned_detections  n_long_gap_fragments  identity_mapping_consistency_vs_full_pct                                                                                                                                                            notes
       yolo11m            1_Full_Fixed_Seat_Identity                                             EXECUTED                    False            False            484093.0                        97.24                 21.0                         98.77                            98.22                    561.0                  12.0                                    100.00 Full Fixed-Seat Identity Resolution baseli

## Step 4: Controlled Seating Arrangement Perturbation Simulation (`ENABLE_SEAT_PERTURBATION_SIMULATION = False`)
*Methodological Scope Disclosure:* Evaluates sensitivity of Fixed-Seat Identity Resolution to controlled perturbations of `student_seats.csv` (spatial anchor jitter `5..50 px`, ROI box scaling `0.50x..1.25x`, row/column pitch compression `15%`, and adjacent seat swaps). Simulation outputs are tagged `CONTROLLED_SIMULATION` and strictly separated from `ACTUAL_OBSERVATION`. This is a geometric sensitivity simulation, **not** an empirical cross-camera or cross-classroom validation.


In [4]:
# ==============================================================================
# CONTROLLED SEATING ARRANGEMENT PERTURBATION SUITE (MULTI-DETECTOR SUPPORT)
# ==============================================================================
if ENABLE_SEAT_PERTURBATION_SIMULATION:
    seats_df = base_identity_module.load_seat_map(str(STUDENT_SEATS_PATH), expected_students=21)
    _all_pert_rows = []
    for _t in get_active_detector_targets(ACTIVE_DETECTOR_MODEL):
        _dkey = _t["detector_key"]
        _det_path = _t["detections_csv"]
        _out_dir = OUTPUT_DIR if _dkey == "yolo11m" else _t["pipeline_dir"]
        _out_dir.mkdir(parents=True, exist_ok=True)
        if not _det_path.exists():
            raise FileNotFoundError(
                f"Detections CSV for '{_dkey}' not found at {_det_path}. Run Notebook 04 with ACTIVE_DETECTOR_MODEL = '{_dkey}' first."
            )
        det_df = pd.read_csv(_det_path)
        perturbation_df = run_controlled_seating_perturbation_suite(
            detections_df=det_df,
            base_seats_df=seats_df,
            base_identity_module=base_identity_module,
            mcdm_eval_fn=evaluate_all_mcdm_methods,
            fps=25.0,
            total_video_frames=23471,
            seed=42,
            enable_simulation=ENABLE_SEAT_PERTURBATION_SIMULATION,
        )
        perturbation_df.insert(0, "detector_model", _dkey)
        if _dkey == "yolo11m":
            perturbation_df.drop(columns=["detector_model"]).to_csv(_out_dir / "controlled_seating_perturbation_simulation.csv", index=False)
        perturbation_df.to_csv(_out_dir / f"controlled_seating_perturbation_simulation_{_dkey}.csv", index=False)
        _all_pert_rows.append(perturbation_df)
        print(f"\n=== Controlled Seating Arrangement Perturbation ({_dkey}) ===")
        print(perturbation_df.to_string(index=False))

    if len(_all_pert_rows) > 1:
        pd.concat(_all_pert_rows, ignore_index=True).to_csv(
            MULTI_DETECTOR_PIPELINE_DIR / "multi_detector_controlled_seating_perturbation_summary.csv", index=False
        )
else:
    logger.info(
        "[LOCKED] Controlled Seating Arrangement Perturbation Suite (%d scenarios, ACTIVE_DETECTOR_MODEL = '%s') "
        "is locked (ENABLE_SEAT_PERTURBATION_SIMULATION = False).",
        len(CONTROLLED_PERTURBATION_SCENARIOS),
        ACTIVE_DETECTOR_MODEL,
    )


=== Controlled Seating Arrangement Perturbation (yolo11m) ===
detector_model                     scenario_id           data_domain         scenario_type  param_value  n_assigned_students  track_assignment_rate_pct  n_assigned_tracks  n_unassigned_tracks  primary_assignment_coverage_pct  mean_student_frame_coverage_pct  mean_seat_purity_preprop_pct  n_unassigned_detections  identity_mapping_consistency_pct  ahp_saw_spearman_rho_vs_baseline                                                                 description
       yolo11m      S0_Baseline_Original_Seats    ACTUAL_OBSERVATION              baseline         0.00                   21                      66.62                932                  467                            97.24                            98.22                         98.77                      561                            100.00                            1.0000                    Original calibrated 21-seat geometry (student_seats.csv)
       yolo11m         